# GeoChange AI — Dataset Inspection (LEVIR-CD Cropped256)

This notebook lists the Hugging Face repository files **before downloading**, downloads its Parquet files, and inspects every sample in small batches. It does not assume Parquet filenames or silently rename dataset columns.

**Outputs**
- Raw Parquet files: `data/raw/`
- Per-sample CSV statistics: `data/processed/stats_<split>.csv`
- One changed-pixel-share histogram per split: `data/processed/`


In [ ]:
# 1. Install the small set of libraries needed for inspection
!pip -q install -U huggingface_hub pyarrow pandas pillow matplotlib tensorflow


In [ ]:
# 2. Runtime, GPU, RAM, TensorFlow and Keras versions
import os
import sys
import subprocess
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

print("\\n--- GPU (nvidia-smi) ---")
gpu_result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
if gpu_result.returncode == 0:
    print(gpu_result.stdout)
else:
    print("nvidia-smi is unavailable or no NVIDIA GPU is attached.")
    print(gpu_result.stderr[:1000])

print("\\n--- RAM ---")
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
    print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
except ImportError:
    print("psutil is not installed; run: !pip install psutil")

import tensorflow as tf
import keras

print("\\n--- ML versions ---")
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Visible TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))


In [ ]:
# 3. List repository files FIRST, then download the Parquet files
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path

REPO_ID = "ericyu/LEVIRCD_Cropped256"
REPO_TYPE = "dataset"

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

api = HfApi()
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)

print(f"Files listed in {REPO_ID}:")
for filename in repo_files:
    print(" -", filename)

# Select Parquet files from the actual repository listing; no filenames are assumed.
parquet_repo_files = sorted(
    filename for filename in repo_files
    if filename.lower().endswith(".parquet")
)

print(f"\\nParquet files found: {len(parquet_repo_files)}")
for filename in parquet_repo_files:
    print(" -", filename)

if len(parquet_repo_files) != 3:
    raise RuntimeError(
        f"Expected the task's three Parquet files, but the repository listing "
        f"contains {len(parquet_repo_files)}. Review the printed listing before proceeding."
    )

downloaded_parquets = []
for repo_filename in parquet_repo_files:
    local_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        filename=repo_filename,
        local_dir=str(raw_dir)
    )
    downloaded_parquets.append(Path(local_path))
    print(f"Downloaded: {repo_filename} -> {local_path}")

print("\\nLocal Parquet files:")
for path in downloaded_parquets:
    print(f" - {path} ({path.stat().st_size / (1024**2):.2f} MB)")


In [ ]:
# 4. Inspect Parquet schemas and identify the split from each actual filename
import pyarrow.parquet as pq

EXPECTED_COLUMNS = {"imageA", "imageB", "label"}

parquet_by_split = {}
for path in downloaded_parquets:
    parquet_file = pq.ParquetFile(path)
    schema = parquet_file.schema_arrow
    row_count = parquet_file.metadata.num_rows

    # Use the filename stem as the split label; do not guess train/val/test names.
    split_name = path.stem
    if split_name in parquet_by_split:
        raise RuntimeError(
            f"Two Parquet files have the same filename stem '{split_name}'. "
            "Give them distinct split labels before continuing."
        )
    parquet_by_split[split_name] = path

    column_names = schema.names
    metadata_columns = [name for name in column_names if name not in EXPECTED_COLUMNS]

    print(f"\\n===== {split_name} =====")
    print("File:", path)
    print("Row count:", row_count)
    print("ALL columns and Arrow dtypes:")
    for field in schema:
        print(f"  {field.name}: {field.type}")

    print("Metadata/extra columns:", metadata_columns if metadata_columns else "None")
    missing = EXPECTED_COLUMNS - set(column_names)
    if missing:
        raise ValueError(f"{split_name} is missing required columns: {sorted(missing)}")


In [ ]:
# 5. Helpers to decode Parquet image values and inspect image properties
from io import BytesIO
from PIL import Image
import numpy as np

def decode_image_value(value):
    """Decode a Parquet/Hugging Face image value into a PIL image.

    Handles raw bytes and the common {bytes, path} image-struct representation.
    This deliberately raises an informative error if the image bytes are absent.
    """
    if value is None:
        raise ValueError("Encountered a null image value.")

    image_bytes = None

    if isinstance(value, dict):
        image_bytes = value.get("bytes")
    elif isinstance(value, (bytes, bytearray, memoryview)):
        image_bytes = bytes(value)
    elif hasattr(value, "as_py"):
        return decode_image_value(value.as_py())

    if image_bytes is None:
        raise ValueError(
            "Image value has no embedded 'bytes'. "
            f"Value type: {type(value).__name__}; value preview: {str(value)[:200]}"
        )

    with Image.open(BytesIO(image_bytes)) as image:
        image.load()
        return image.copy()

def image_check(image, expected_mode, expected_size=(256, 256)):
    """Return True only when both dimensions and image mode match."""
    return image.size == expected_size and image.mode == expected_mode

def mask_changed_share(mask_image):
    """Return changed-pixel fraction and unique mask values.

    A pixel is treated as changed when its decoded grayscale value is nonzero.
    The expected label encoding is L with values 0 and 255; unexpected values
    are reported rather than silently remapped.
    """
    mask_array = np.asarray(mask_image)
    unique_values = np.unique(mask_array)
    changed_pixels = int(np.count_nonzero(mask_array > 0))
    total_pixels = int(mask_array.size)
    share = changed_pixels / total_pixels if total_pixels else float("nan")
    return share, changed_pixels, total_pixels, unique_values.tolist()


In [ ]:
# 6. Iterate every row in small batches; calculate per-sample and per-split statistics
import pandas as pd
import math
from collections import Counter

BATCH_SIZE = 32  # Keep memory use low; only this many rows are decoded at a time.

split_summaries = {}

for split_name, parquet_path in parquet_by_split.items():
    parquet_file = pq.ParquetFile(parquet_path)

    sample_rows = []
    all_mask_values = set()
    bucket_counts = Counter({
        "0%": 0,
        ">0% and <5%": 0,
        "5-20% (inclusive)": 0,
        ">20%": 0
    })

    imageA_bad = 0
    imageB_bad = 0
    mask_bad = 0
    total_rows = 0
    empty_masks = 0

    for record_batch in parquet_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=["imageA", "imageB", "label"]
    ):
        # A batch is converted to Python records, processed, then discarded.
        for row in record_batch.to_pylist():
            row_index = total_rows

            image_a = decode_image_value(row["imageA"])
            image_b = decode_image_value(row["imageB"])
            mask = decode_image_value(row["label"])

            imageA_ok = image_check(image_a, "RGB")
            imageB_ok = image_check(image_b, "RGB")
            mask_ok = image_check(mask, "L")

            imageA_bad += int(not imageA_ok)
            imageB_bad += int(not imageB_ok)
            mask_bad += int(not mask_ok)

            share, changed_pixels, total_pixels, unique_values = mask_changed_share(mask)
            all_mask_values.update(unique_values)

            if changed_pixels == 0:
                empty_masks += 1
                bucket_counts["0%"] += 1
            elif share < 0.05:
                bucket_counts[">0% and <5%"] += 1
            elif share <= 0.20:
                bucket_counts["5-20% (inclusive)"] += 1
            else:
                bucket_counts[">20%"] += 1

            sample_rows.append({
                "sample_index": row_index,
                "changed_pixel_share": share,
                "changed_pixel_percent": share * 100.0,
                "changed_pixels": changed_pixels,
                "total_mask_pixels": total_pixels,
                "mask_unique_values": ",".join(map(str, unique_values)),
                "imageA_is_256x256_RGB": imageA_ok,
                "imageB_is_256x256_RGB": imageB_ok,
                "label_is_256x256_L": mask_ok
            })

            total_rows += 1

    stats_df = pd.DataFrame(sample_rows)
    csv_path = processed_dir / f"stats_{split_name}.csv"
    stats_df.to_csv(csv_path, index=False)

    shares = stats_df["changed_pixel_share"]
    summary = {
        "rows": total_rows,
        "mean_share": float(shares.mean()),
        "median_share": float(shares.median()),
        "min_share": float(shares.min()),
        "max_share": float(shares.max()),
        "empty_masks": empty_masks,
        "bucket_counts": dict(bucket_counts),
        "mask_unique_values": sorted(all_mask_values),
        "imageA_bad": imageA_bad,
        "imageB_bad": imageB_bad,
        "mask_bad": mask_bad,
        "csv_path": str(csv_path)
    }
    split_summaries[split_name] = summary

    print(f"\\n===== FULL STATISTICS: {split_name} =====")
    print("Rows processed:", total_rows)
    print("Mask unique values found across split:", summary["mask_unique_values"])
    print(f"Changed-pixel share mean:   {summary['mean_share']:.6f} ({summary['mean_share']*100:.4f}%)")
    print(f"Changed-pixel share median: {summary['median_share']:.6f} ({summary['median_share']*100:.4f}%)")
    print(f"Changed-pixel share min:    {summary['min_share']:.6f} ({summary['min_share']*100:.4f}%)")
    print(f"Changed-pixel share max:    {summary['max_share']:.6f} ({summary['max_share']*100:.4f}%)")
    print("Number of empty masks:", empty_masks)
    print("Exclusive bucket counts (each sample belongs to exactly one bucket):")
    for bucket, count in bucket_counts.items():
        print(f"  {bucket}: {count}")
    print("Image/mask validation failures:")
    print(f"  imageA not 256x256 RGB: {imageA_bad}")
    print(f"  imageB not 256x256 RGB: {imageB_bad}")
    print(f"  label not 256x256 L:   {mask_bad}")
    print("Per-sample CSV:", csv_path)


In [ ]:
# 7. Plot one histogram of changed-pixel share for each split
import matplotlib.pyplot as plt

for split_name in parquet_by_split:
    csv_path = processed_dir / f"stats_{split_name}.csv"
    stats_df = pd.read_csv(csv_path)

    plt.figure(figsize=(8, 5))
    plt.hist(stats_df["changed_pixel_share"], bins=50)
    plt.xlabel("Changed-pixel share (fraction of mask pixels)")
    plt.ylabel("Number of samples")
    plt.title(f"Changed-pixel share distribution — {split_name}")
    plt.tight_layout()

    plot_path = processed_dir / f"hist_changed_pixel_share_{split_name}.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print("Saved histogram:", plot_path)


In [ ]:
# 8. Compact paste-back summary
print("\\n========== GEOCHANGE AI — INSPECTION SUMMARY ==========")
print("Dataset repo:", REPO_ID)
print("Parquet files downloaded:", len(downloaded_parquets))
for split_name, summary in split_summaries.items():
    print(
        f"{split_name}: rows={summary['rows']}, "
        f"mask_values={summary['mask_unique_values']}, "
        f"share_mean={summary['mean_share']:.5f}, "
        f"share_median={summary['median_share']:.5f}, "
        f"share_min={summary['min_share']:.5f}, "
        f"share_max={summary['max_share']:.5f}, "
        f"empty_masks={summary['empty_masks']}, "
        f"buckets={summary['bucket_counts']}, "
        f"invalid(A/B/mask)={summary['imageA_bad']}/{summary['imageB_bad']}/{summary['mask_bad']}"
    )

all_checks_passed = all(
    s["imageA_bad"] == 0 and s["imageB_bad"] == 0 and s["mask_bad"] == 0
    for s in split_summaries.values()
)
print("All image/mask format checks passed:", all_checks_passed)
print("Per-sample CSVs and histograms saved under:", processed_dir)
print("Bucket definitions are exclusive: 0%; >0% and <5%; 5–20% inclusive; >20%.")
print("=======================================================")
